# Spatial Detector Comparison — per-scenario

Compares **DSM** (our global), **CF-Attn / CF-Attn-CFAR / NeighborMLP** (spatial DSMs),
**AMF** + **AMF-local**, **CEM** + **CEM-local**, **GMM-Levin**.

Each scenario has its own cell below — edit its `OVERRIDES` dict to change the
config for **that scenario only**, then run the cell to train + score + plot.

Scenarios: **0–3** = manual boxes, **4–7** = random boxes. Deep nets train on **GPU**.

In [ ]:
# ── Cell 1: Clone repo (pinned branch) + install deps ─────────────
import os, sys

REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'
BRANCH        = 'iid-unified-experiment'   # spatial compare code lives here

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    !git -C {LOCAL_PROJECT} fetch --all -q
    !git -C {LOCAL_PROJECT} checkout {BRANCH} -q
    !git -C {LOCAL_PROJECT} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL_PROJECT}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml pandas

for p in [LOCAL_PROJECT, os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(LOCAL_PROJECT)
print('Ready. CWD:', os.getcwd())
!git -C {LOCAL_PROJECT} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — enable Runtime > Change runtime type > GPU for fast deep nets.')
print('PyTorch', torch.__version__, ' device =', DEVICE)

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ──────────────────────
import os
RESULTS = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/compare_results'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                    '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
            if os.path.exists(src):
                os.symlink(src, DST); print('Linked dataset from', src); break
        else:
            print('WARNING: pavia-u.mat not found on Drive.')
    else:
        print('Dataset already linked.')
except Exception as e:
    print('Drive not mounted:', repr(e))
os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing!'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Helpers — run one scenario with custom config + show results ──
import subprocess, sys, os, glob, json, tempfile
import yaml
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.image as mpimg
from IPython.display import display

QUICK = False   # True = fast smoke test (few epochs) for ALL scenario cells

# Base config shared by every scenario (per-scenario OVERRIDES are merged on top).
BASE_CFG = yaml.safe_load(open('experiments/spatial/colab.yaml')) or {}

# Editable defaults you may want to tweak per scenario — see keys in run_compare.py
#   amplitude, target_fraction, n_budget, k, local_scm_loading,
#   cfattn_epochs, nmlp_epochs, dsm_epochs, cfattn_K, nmlp_K, ...

def _latest_run_for(scenario_idx):
    cands = []
    for r in glob.glob(os.path.join(RESULTS, 'compare_*')):
        mp = os.path.join(r, 'metrics.json')
        if os.path.exists(mp):
            try:
                if json.load(open(mp)).get('scenario_index') == scenario_idx:
                    cands.append(r)
            except Exception:
                pass
    return max(cands) if cands else None

def show_results(run_dir):
    if run_dir is None:
        print('No run dir found.'); return
    print('Run:', run_dir)
    csv = os.path.join(run_dir, 'summary_table.csv')
    if os.path.exists(csv):
        df = pd.read_csv(csv)
        display(df.style
                .background_gradient(subset=['pAUC@0.05', 'AUC', 'Pd@Pfa=0.05'], cmap='Greens')
                .background_gradient(subset=['Pfa_avg', 'Pfa_max'], cmap='Reds')
                .format({c: '{:.3f}' for c in df.columns if c != 'Detector'}))
    for name in ['false_color', 'label_map', 'detection_maps', 'roc', 'pfa_per_class']:
        png = os.path.join(run_dir, name + '.png')
        if os.path.exists(png):
            img = mpimg.imread(png)
            fig, ax = plt.subplots(figsize=(12, 8))
            ax.imshow(img); ax.axis('off'); ax.set_title(name, fontsize=10)
            plt.tight_layout(); plt.show()

def run_scenario(scenario_idx, overrides=None, show=True):
    cfg = dict(BASE_CFG)
    cfg.update(overrides or {})
    cfg['scenario_index'] = int(scenario_idx)
    fd, tmp = tempfile.mkstemp(suffix='.yaml', prefix=f'cfg_s{scenario_idx}_')
    os.close(fd)
    yaml.safe_dump(cfg, open(tmp, 'w'))
    cmd = [sys.executable, '-u', 'experiments/spatial/run_compare.py',
           '--config', tmp, '--results_dir', RESULTS, '--scenario', str(scenario_idx)]
    if QUICK:
        cmd.append('--dry-run')
    print('Running scenario', scenario_idx, '| overrides:', overrides or {}, flush=True)
    subprocess.run(cmd, check=True)
    run_dir = _latest_run_for(scenario_idx)
    if show:
        show_results(run_dir)
    return run_dir

print('Helpers ready.  QUICK =', QUICK)

## Scenario 0 — manual box 0

In [ ]:
# ── Scenario 0: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_0 = run_scenario(0, OVERRIDES)

## Scenario 1 — manual box 1

In [ ]:
# ── Scenario 1: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_1 = run_scenario(1, OVERRIDES)

## Scenario 2 — manual box 2

In [ ]:
# ── Scenario 2: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_2 = run_scenario(2, OVERRIDES)

## Scenario 3 — manual box 3

In [ ]:
# ── Scenario 3: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_3 = run_scenario(3, OVERRIDES)

## Scenario 4 — random box (seed 42)

In [ ]:
# ── Scenario 4: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_4 = run_scenario(4, OVERRIDES)

## Scenario 5 — random box (seed 123)

In [ ]:
# ── Scenario 5: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_5 = run_scenario(5, OVERRIDES)

## Scenario 6 — random box (seed 456)

In [ ]:
# ── Scenario 6: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_6 = run_scenario(6, OVERRIDES)

## Scenario 7 — random box (seed 789)

In [ ]:
# ── Scenario 7: edit OVERRIDES (config for THIS scenario only) ──
OVERRIDES = dict(
    # amplitude       = 0.15,
    # target_fraction = 0.10,
    # n_budget        = 2000,
    # k               = 5,
    # local_scm_loading = 0.1,
    # cfattn_epochs   = 300,
    # nmlp_epochs     = 300,
    # dsm_epochs      = 1000,
)
run_dir_7 = run_scenario(7, OVERRIDES)

## Aggregate across all run scenarios (mean)

In [ ]:
# ── Aggregate: mean of each metric across whichever scenarios you ran ──
by_scen = {}
for r in sorted(glob.glob(os.path.join(RESULTS, 'compare_*'))):
    mp = os.path.join(r, 'metrics.json')
    if os.path.exists(mp):
        m = json.load(open(mp))
        by_scen[m['scenario_index']] = m   # latest run per scenario
print('Scenarios aggregated:', sorted(by_scen))

metrics = ['pAUC@0.05', 'AUC', 'Pd@Pfa=0.05', 'Pfa_avg', 'Pfa_max']
if by_scen:
    dets = [row['Detector'] for row in next(iter(by_scen.values()))['rows']]
    acc = {d: {mt: [] for mt in metrics} for d in dets}
    for m in by_scen.values():
        for row in m['rows']:
            for mt in metrics:
                acc[row['Detector']][mt].append(float(row[mt]))
    rows = []
    for d in dets:
        rec = {'Detector': d, 'n_scen': len(acc[d]['AUC'])}
        for mt in metrics:
            rec[mt] = float(np.nanmean(acc[d][mt]))
        rows.append(rec)
    df = pd.DataFrame(rows)
    display(df.style
            .background_gradient(subset=['pAUC@0.05', 'AUC', 'Pd@Pfa=0.05'], cmap='Greens')
            .background_gradient(subset=['Pfa_avg', 'Pfa_max'], cmap='Reds')
            .format({c: '{:.3f}' for c in metrics}))
    out_csv = os.path.join(RESULTS, 'summary_all_scenarios.csv')
    df.to_csv(out_csv, index=False)
    print('Saved:', out_csv)
else:
    print('No scenario results found yet — run some scenario cells first.')